In [ ]:
from google.colab import drive
import sys
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Add the "Colab Notebooks" folder to the system path
# This allows 'import data_aug' to work
module_path = '/content/drive/MyDrive/Colab_Notebooks'
if module_path not in sys.path:
    sys.path.append(module_path)

# 3. Test the import
import data_aug
print("Success! data_aug is now accessible.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Success! data_aug is now accessible.


In [ ]:
import os

# Create the hidden kaggle directory
!mkdir -p ~/.kaggle

# Copy kaggle.json from your specific Drive folder to the root .kaggle folder
!cp "/content/drive/MyDrive/Colab_Notebooks/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the data
!kaggle competitions download -c hms-harmful-brain-activity-classification

# Create dataset directory and unzip
!mkdir -p dataset
!unzip -q hms-harmful-brain-activity-classification.zip -d dataset/

# Clean up
!rm hms-harmful-brain-activity-classification.zip

print("Loading success! Data is ready in /content/dataset")

100% 18.4G/18.4G [01:01<00:00, 375MB/s]
100% 18.4G/18.4G [01:01<00:00, 320MB/s]


In [3]:
import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
import torch.nn.functional as F
from tqdm.auto import tqdm

# Optional: Install timm if not available for EfficientNet backbones
try:
    import timm
except ImportError:
    !pip install timm
    import timm

In [4]:
class CFG:
    seed = 42
    batch_size = 4 # Reduced for memory safety with CWT
    use_amp = True
    epochs = 5
    lr = 1e-3
    n_fold = 10
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # PATHS
    train_csv = '/content/drive/MyDrive/Colab_Notebooks/confident_train.csv'
    test_csv = '/content/drive/MyDrive/Colab_Notebooks/confident_test.csv'
    # Important: Both confident_train and confident_test entries are likely in 'train_eegs'
    eeg_path = '/content/dataset/train_eegs'

    # Set this to True to test without the actual .parquet files
    USE_DUMMY = False

    # CWT Parameters
    wavelet_width = 8.0
    fs = 200.0
    lower_freq = 0.5
    upper_freq = 40.0
    n_scales = 128

    # Montage: Standard 8-channel differential montage
    FEATS = ['Fp1','T3','T5','O1','Fp2','T4','T6','O2']

    # Stronger Augmentation Settings
    use_mixup = True
    mixup_prob = 0.8        # Increase from 0.6
    mixup_alpha = 2.0       # Increase from 0.4 for more diverse mixes

    time_mask_prob = 0.7    # Increase from 0.35
    time_mask_frac_min = 0.05
    time_mask_frac_max = 0.20 # Mask up to 20% of the signal

    channel_drop_prob = 0.5 # Increase from 0.25
    channel_drop_max = 4    # Drop up to 4 of the 8 channels

    # Gaussian Noise and Scaling
    aug_noise_std_max = 0.01
    aug_scale_min = 0.8
    aug_scale_max = 1.2

    # Optimizer Regularization
    # If using AdamW, ensure weight_decay is active
    weight_decay = 0.1

    # Transformer Settings
    embed_dim = 128  # Dimension of the Transformer hidden states
    nhead = 4        # Number of attention heads
    num_layers = 2   # Number of Transformer blocks
    dropout = 0.1

In [5]:
class CWT(nn.Module):
    def __init__(self, wavelet_width, fs, lower_freq, upper_freq, n_scales,
                 size_factor=1.0, border_crop=0, stride=1):
        super().__init__()
        self.initial_wavelet_width = wavelet_width
        self.fs = fs
        self.lower_freq = lower_freq
        self.upper_freq = upper_freq
        self.size_factor = size_factor
        self.n_scales = n_scales
        self.wavelet_width = wavelet_width
        self.border_crop = border_crop
        self.stride = stride

        wavelet_bank_real, wavelet_bank_imag = self._build_wavelet_kernel()
        self.wavelet_bank_real = nn.Parameter(wavelet_bank_real.float(), requires_grad=False)
        self.wavelet_bank_imag = nn.Parameter(wavelet_bank_imag.float(), requires_grad=False)
        self.kernel_size = self.wavelet_bank_real.size(3)

    def _build_wavelet_kernel(self):
        s_0 = 1 / self.upper_freq
        s_n = 1 / self.lower_freq
        base = np.power(s_n / s_0, 1 / (self.n_scales - 1))
        scales = s_0 * np.power(base, np.arange(self.n_scales))

        truncation_size = scales.max() * np.sqrt(4.5 * self.initial_wavelet_width) * self.fs
        one_side = int(self.size_factor * truncation_size)
        kernel_size = 2 * one_side + 1
        k_array = np.arange(kernel_size, dtype=np.float32) - one_side
        t_array = k_array / self.fs

        wavelet_bank_real, wavelet_bank_imag = [], []
        for scale in scales:
            norm_constant = np.sqrt(np.pi * self.wavelet_width) * scale * self.fs / 2.0
            scaled_t = t_array / scale
            exp_term = np.exp(-(scaled_t ** 2) / self.wavelet_width)
            wavelet_bank_real.append(exp_term / norm_constant * np.cos(2 * np.pi * scaled_t))
            wavelet_bank_imag.append(exp_term / norm_constant * np.sin(2 * np.pi * scaled_t))

        # Shape: [Scales, 1, 1, Kernel]
        wavelet_bank_real = torch.from_numpy(np.stack(wavelet_bank_real)).unsqueeze(1).unsqueeze(2)
        wavelet_bank_imag = torch.from_numpy(np.stack(wavelet_bank_imag)).unsqueeze(1).unsqueeze(2)
        return wavelet_bank_real, wavelet_bank_imag

    def forward(self, x):
        # x: [Batch, Channels, Time]
        B, C, T = x.shape

        # Flatten Batch and Channels to process everything in ONE convolution call
        # New shape: [B*C, 1, 1, T]
        x = x.view(B * C, 1, 1, T)

        in_width = T
        out_width = int(np.ceil(in_width / self.stride))
        pad_along_width = max((out_width - 1) * self.stride + self.kernel_size - in_width, 0)
        padding = int(pad_along_width // 2 + 1)

        # Vectorized Convolution
        out_real = F.conv2d(x, self.wavelet_bank_real, stride=(1, self.stride), padding=(0, padding))
        out_imag = F.conv2d(x, self.wavelet_bank_imag, stride=(1, self.stride), padding=(0, padding))

        # Reshape back to [Batch, Channels, Scales, Time_reduced]
        # out_real is [B*C, Scales, 1, T_reduced]
        out_real = out_real.view(B, C, self.n_scales, -1)
        out_imag = out_imag.view(B, C, self.n_scales, -1)

        # Magnitudes
        scalograms = torch.sqrt(out_real ** 2 + out_imag ** 2)
        return scalograms

In [ ]:
class HMSDataset(Dataset):
    def __init__(self, df, path, mode='train'):
        self.df = df
        self.path = path
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        eeg_id = row['eeg_id']
        offset = int(row['eeg_label_offset_seconds'] * 200)

        # Load Parquet
        eeg_path = os.path.join(self.path, f"{eeg_id}.parquet")
        eeg_data = pd.read_parquet(eeg_path)

        # Slice 50 seconds (10,000 samples)
        data = eeg_data.iloc[offset : offset + 10000]

        # Feature Engineering
        X = data[CFG.FEATS].values.T
        X = np.nan_to_num(X, nan=0.0)

        # Standardize
        X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-6)

        vote_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

        if self.mode == 'train':
            X = data_aug.augment_sample_np(X, CFG, target_sample_rate=200)
            # 1. Get raw votes
            votes_raw = row[vote_cols].values.astype('float32')

            # 2. Calculate Total Votes (Required for your data_aug script)
            total_votes = votes_raw.sum()

            # 3. Create probability distribution (y)
            y = votes_raw / (total_votes + 1e-6)

            # RETURN THREE ITEMS: X, y, and total_votes
            return (
                torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(total_votes, dtype=torch.float32)
            )
        elif self.mode == 'val':
            votes_raw = row[vote_cols].values.astype('float32')
            y = votes_raw / (votes_raw.sum() + 1e-6)
            return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

        # Final Submission (Hidden Test set has no vote columns)
        else:
            return torch.tensor(X, dtype=torch.float32)

In [6]:
class HMSModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cwt = CWT(
            wavelet_width=CFG.wavelet_width, fs=CFG.fs,
            lower_freq=CFG.lower_freq, upper_freq=CFG.upper_freq,
            n_scales=CFG.n_scales, stride=4
        )

        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=True,
            in_chans=len(CFG.FEATS),
            features_only=True
        )

        # This line automatically detects that the last stage has 1280 channels
        last_layer_channels = self.backbone.feature_info[-1]['num_chs']

        self.feature_proj = nn.Conv2d(last_layer_channels, CFG.embed_dim, kernel_size=1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=CFG.embed_dim,
            nhead=CFG.nhead,
            dim_feedforward=CFG.embed_dim * 4,
            dropout=CFG.dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=CFG.num_layers)
        self.classifier = nn.Linear(CFG.embed_dim, 6)

    def forward(self, x):
        scalogram = self.cwt(x)
        # features is a list; [-1] is the final feature map
        features = self.backbone(scalogram)[-1]

        x = self.feature_proj(features)
        x = torch.mean(x, dim=2) # Collapse frequency
        x = x.permute(0, 2, 1)   # [Batch, Time, Embed]

        x = self.transformer(x)
        x = torch.mean(x, dim=1) # Global pooling over time
        return self.classifier(x)

In [ ]:
import gc
import torch
from data_aug import build_train_collate_fn

# Clear memory before starting
gc.collect()
torch.cuda.empty_cache()

# 1. Initialize Model, Optimizer, and the MISSING Scaler
model = HMSModel().to(CFG.device)
optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
criterion = nn.KLDivLoss(reduction='batchmean')

# Fix: Initialize the scaler here
scaler = torch.amp.GradScaler('cuda', enabled=CFG.use_amp)
gkf = GroupKFold(n_splits=CFG.n_fold)

train_collate = build_train_collate_fn(CFG)
train_df = pd.read_csv(CFG.train_csv)

# 10% Patient-based split
gkf = GroupKFold(n_splits=CFG.n_fold)
for train_idx, val_idx in gkf.split(train_df, groups=train_df['patient_id']):
    train_split = train_df.iloc[train_idx].reset_index(drop=True)
    val_split = train_df.iloc[val_idx].reset_index(drop=True)
    break

train_loader = DataLoader(HMSDataset(train_df.iloc[train_idx], CFG.eeg_path),
                          batch_size=CFG.batch_size, shuffle=True, collate_fn=train_collate, num_workers=0)
val_loader = DataLoader(HMSDataset(val_split, CFG.eeg_path), batch_size=CFG.batch_size, shuffle=False, num_workers=2)

print(f"Starting training on {CFG.device} with Mixed Precision...")

for epoch in range(CFG.epochs):
    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for X, y, votes in pbar:
        X, y = X.to(CFG.device), y.to(CFG.device)
        optimizer.zero_grad()

        # Modern autocast syntax to avoid FutureWarnings
        with torch.amp.autocast('cuda', enabled=CFG.use_amp):
            output = model(X)
            loss = criterion(F.log_softmax(output, dim=1), y)

        # Scaled Backpropagation
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # --- Validation ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]")
        for X, y, votes in val_pbar:
            X, y = X.to(CFG.device), y.to(CFG.device)
            with torch.amp.autocast('cuda', enabled=CFG.use_amp):
                output = model(X)
                loss = criterion(F.log_softmax(output, dim=1), y)
            val_loss += loss.item()
            val_pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_val_loss = val_loss / len(val_loader)
    if 'best_val_loss' not in locals() or avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"!!! Best Model Saved (Loss: {best_val_loss:.4f})")

    print(f"\nEpoch {epoch+1} Summary: Train Loss {train_loss/len(train_loader):.4f} | Val Loss {avg_val_loss:.4f}")

Starting training on cuda with Mixed Precision...


Epoch 1:   0%|          | 0/13748 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/1528 [00:00<?, ?it/s]

!!! Best Model Saved (Loss: 1.6449)

Epoch 1 Summary: Train Loss 1.3344 | Val Loss 1.6449


Epoch 2:   0%|          | 0/13748 [00:00<?, ?it/s]

...

Epoch 4 [Val]:   0%|          | 0/1528 [00:00<?, ?it/s]


Epoch 4 Summary: Train Loss 1.3270 | Val Loss 1.6611


Epoch 5:   0%|          | 0/13748 [00:00<?, ?it/s]

Epoch 5 [Val]:   0%|          | 0/1528 [00:00<?, ?it/s]


Epoch 5 Summary: Train Loss 1.3305 | Val Loss 1.6566


In [ ]:
# test

import torch.nn.functional as F

# 1. Load the test dataframe
test_df = pd.read_csv(CFG.test_csv)

# 2. Prepare DataLoader
# Note: mode='test' in your HMSDataset only returns the EEG tensor X
test_dataset = HMSDataset(test_df, CFG.eeg_path, mode='val')
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=2
)

# 3. Load Model and Weights
model = HMSModel().to(CFG.device)
model.load_state_dict(torch.load('best_model.pth', map_location=CFG.device))
model.eval()

# 4. Inference loop
test_loss = 0
preds = []
with torch.no_grad():
    for X, y in tqdm(test_loader, desc="Evaluating"): # Now expects 2 values
        X, y = X.to(CFG.device), y.to(CFG.device)

        output = model(X)

        # KL Divergence Calculation
        # log_softmax(output) is Q, y is P
        loss = F.kl_div(F.log_softmax(output, dim=1), y, reduction='batchmean')
        test_loss += loss.item() * X.size(0)

        preds.append(F.softmax(output, dim=1).cpu().numpy())

print(f"Total Test KL Loss: {test_loss / len(test_loader.dataset):.4f}")

# 5. Concatenate all predictions
predictions = np.concatenate(preds, axis=0)

# 6. Create Submission DataFrame
# The order is guaranteed to be the same as test_csv because shuffle=False
target_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
sub_df = pd.DataFrame({'eeg_id': test_df['eeg_id']})
sub_df[target_cols] = predictions

# 7. Save and Display
sub_df.to_csv('submission.csv', index=False)
print("Inference Complete! Results saved to submission.csv")
display(sub_df.head())

Evaluating: 100%|██████████| 1656/1656 [03:36<00:00,  7.67it/s]

Total Test KL Loss: 1.5389
Inference Complete! Results saved to submission.csv


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,4245882082,0.289923,0.129505,0.103548,0.106300,0.197619,0.173105
1,2857590162,0.282856,0.126977,0.097589,0.107846,0.196902,0.187830
2,2857590162,0.282856,0.126977,0.097589,0.107846,0.196902,0.187830
3,2857590162,0.290090,0.129359,0.103377,0.106242,0.197606,0.173325
4,3523834378,0.288896,0.130405,0.104606,0.106650,0.197697,0.171747


In [2]:
!pip install torch-summary
!pip install torchinfo

In [7]:
from torchinfo import summary
import torch

# Initialize model and move to device
model = HMSModel().to(CFG.device)

# Setting depth=2 will show the main blocks (Level 1)
# and their immediate children (Level 2), but hide the
# granular operations like Identity, SiLU, or Flatten (Level 3).
summary(
    model,
    input_size=(1, 8, 10000),
    device=CFG.device,
    depth=2,  # <--- This is the magic line
    col_names=["input_size", "output_size", "num_params", "kernel_size"],
    row_settings=["var_names"]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Layer (type (var_name))                                 Input Shape               Output Shape              Param #                   Kernel Shape
HMSModel (HMSModel)                                     [1, 8, 10000]             [1, 6]                    --                        --
├─CWT (cwt)                                             [1, 8, 10000]             [1, 8, 128, 2500]         (1,228,544)               4799
├─EfficientNetFeatures (backbone)                       [1, 8, 128, 2500]         [1, 16, 64, 1250]         --                        --
│    └─Conv2d (conv_stem)                               [1, 8, 128, 2500]         [1, 32, 64, 1250]         2,304                     [3, 3]
│    └─BatchNormAct2d (bn1)                             [1, 32, 64, 1250]         [1, 32, 64, 1250]         64                        --
│    └─Sequential (blocks)                              --                        --                        3,594,460                 --
├─Conv2d (feature_proj)  